# VEDJE: index once, rerank every query

VEDJE reranks the candidates of a first-stage video retriever with a 33M-parameter joint encoder. Each video is encoded once into a small cache of a few tokens per sampled frame. A query never waits for a video to be encoded.

In about two minutes on a free CPU runtime, this notebook:

1. indexes a video, the sample clip or one of yours, into its cache;
2. shows what the frozen encoder follows from frame to frame, which the cache keeps frame by frame;
3. times a query on this runtime: VEDJE reading its candidates' caches against a typical reranker that encodes the candidates again.

Choose **Runtime > Run all** to start. [Project page](https://gabrieleserussi.github.io/vedje/) · [Code](https://github.com/GabrieleSerussi/vedje)

In [ ]:
#@title 1. Set up and choose a video
video = "The sample clip (a peacock)"  #@param ["The sample clip (a peacock)", "Upload a video", "A video URL or path"]
video_url = ""  #@param {type:"string"}
cache_size = "64 tokens (48 KiB)"  #@param ["64 tokens (48 KiB)", "16 tokens (12 KiB)"]
#@markdown **Upload a video** opens a file picker when the cell runs; **A video URL or path** reads `video_url`. The first run installs the repository and downloads the frozen encoder, which takes about a minute.

import importlib.metadata, importlib.util, io, os, subprocess, sys, tempfile, time, urllib.request, warnings
start = time.time()
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
os.environ.setdefault("HF_HUB_VERBOSITY", "error")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
warnings.filterwarnings("ignore")
REPO = os.environ.get("VEDJE_REPO", "/content/vedje")
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/GabrieleSerussi/vedje.git", REPO], check=True)

def version_of(name):
    try:
        return tuple(int(part) for part in importlib.metadata.version(name).split(".")[:2])
    except Exception:
        return (0, 0)

needed = []
if version_of("transformers") < (5, 13):
    needed.append("transformers>=5.13")  # the first release with the VideoPrism models
if importlib.util.find_spec("ruamel") is None:
    needed.append("ruamel.yaml")
if needed:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *needed], check=True)

sys.path.insert(0, REPO)
os.environ["PYTHONPATH"] = REPO + os.pathsep + os.environ.get("PYTHONPATH", "")
import torch
import matplotlib.pyplot as plt
from IPython.display import HTML, Image as ShowImage, display
from vedje import cache, config, features, model

BLUE, GRAY, INK = "#2548F6", "#6A6A6A", "#1F2937"
M = 4 if cache_size.startswith("64") else 1  # tokens per sampled frame
device = "cuda" if torch.cuda.is_available() else "cpu"
SAMPLE = os.path.join(REPO, "colab", "sample")

def cards(*items):
    """A row of number cards, each (label, value, note, colour)."""
    card = ('<div style="flex:1 1 190px;margin:4px;padding:12px 14px;border-radius:12px;background:#F4F6FB;color:#1F2937;'
            'font-family:system-ui,-apple-system,sans-serif"><div style="font-size:11px;letter-spacing:.08em;'
            'text-transform:uppercase;color:#6B7280">{}</div><div style="margin:4px 0 2px;font-size:22px;font-weight:600;'
            'color:{}">{}</div><div style="font-size:12.5px;line-height:1.35;color:#4B5563">{}</div></div>')
    display(HTML('<div style="display:flex;flex-wrap:wrap;margin:6px 0">'
                 + "".join(card.format(label, colour, value, note) for label, value, note, colour in items) + "</div>"))

def fetch(url):
    local = os.path.join(tempfile.gettempdir(), "my_video" + (os.path.splitext(url.split("?")[0])[1] or ".mp4"))
    request = urllib.request.Request(url, headers={"User-Agent": "vedje-colab-tour/1.0 (https://github.com/GabrieleSerussi/vedje)"})
    with urllib.request.urlopen(request) as response, open(local, "wb") as out:
        out.write(response.read())
    return local

source, name = SAMPLE, "the sample clip"
if video.startswith("Upload"):
    try:
        from google.colab import files
        uploaded = files.upload()
        if uploaded:
            name = next(iter(uploaded))
            source = os.path.abspath(name)
    except ImportError:
        print("Uploading needs Colab, so this run uses the sample clip.")
elif video.startswith("A video URL"):
    if video_url.startswith(("http://", "https://")):
        source, name = fetch(video_url), "your video"
    elif video_url and os.path.exists(video_url):
        source, name = video_url, os.path.basename(video_url)
    else:
        print("Fill in video_url with a URL or a path; this run uses the sample clip.")
if source != SAMPLE and importlib.util.find_spec("av") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "av"], check=True)
frames = features.load_frames(source)  # 16 frames, 3 x 288 x 288, values in [0, 1]

fig, axes = plt.subplots(2, 8, figsize=(12, 3.6), gridspec_kw={"hspace": 0.32, "wspace": 0.06})
for t, ax in enumerate(axes.flat):
    ax.imshow(frames[t].clamp(0, 1).permute(1, 2, 0).numpy())
    ax.set_title(str(t + 1), fontsize=9, color="#555555", pad=3)
    ax.axis("off")
fig.suptitle(f"The 16 frames sampled from {name}", fontsize=11)
plt.show()
print(f"Ready in {time.time() - start:.0f} s, on {'a GPU' if device == 'cuda' else 'the CPU'}.")

## 2. Index the video once

The frozen [VideoPrism-B](https://proceedings.mlr.press/v235/zhao24f.html) encoder reads the 16 sampled frames once, when the video is indexed. VEDJE's compressor then turns the 256 patch features of each frame into a few tokens and keeps them in temporal order. This cache is all that a query reads later.

The release has no trained weights, so the compressor here is freshly initialised. Its cache has the shape, the order and the size of a trained one.

In [ ]:
#@title 2. Index the video: encode it once and write its cache
backbone = features.load_backbone(device=device)
start = time.time()
patches, _ = features.extract_patch_features(backbone, frames)  # (1, 16 * 256, 768): the frozen patch features
if device == "cuda":
    torch.cuda.synchronize()
encode_s = time.time() - start
torch.manual_seed(0)
compressor = cache.FramewiseCrossAttentionCompressor(num_queries_per_frame=M).eval().to(device)
with torch.inference_mode():
    Z = compressor(patches)  # the cache Z(v): (1, 16 * M, 384)
stored, full = cache.cache_bytes(16, M, 384), cache.cache_bytes(16, 257, 768)  # Table 9: 256 patches and one frame vector
cards(("Encoded once", f"{encode_s:.1f} s", f"the frozen encoder reads the 16 frames on {'a GPU' if device == 'cuda' else 'the CPU'}, at indexing time", INK),
      ("The cache", f"{16 * M} tokens", f"{M} per sampled frame, kept in temporal order", BLUE),
      ("Stored per video", f"{stored / 1024:g} KiB", f"{full / stored:.1f} times less than the frame-and-patch features ({full / 2**20:.2f} MiB)", BLUE))

# the cache, frame by frame
from PIL import Image
steps = []
for t in range(16):
    fig = plt.figure(figsize=(9, 2.9), dpi=80)
    ax_f = fig.add_axes([0.01, 0.12, 0.27, 0.76]); ax_c = fig.add_axes([0.33, 0.12, 0.66, 0.76])
    ax_f.imshow(frames[t].clamp(0, 1).permute(1, 2, 0).numpy()); ax_f.set_title(f"Frame {t + 1} of 16", fontsize=10); ax_f.axis("off")
    for i in range(16):  # one group of M tokens per frame, in temporal order
        for j in range(M):
            ax_c.add_patch(plt.Rectangle((i + 0.1, j * (0.8 / M) + 0.1), 0.8, 0.8 / M - 0.06,
                                         color=BLUE if i <= t else "#E3E3E3", alpha=1.0 if i == t else 0.75))
        ax_c.text(i + 0.5, -0.25, f"Z{i + 1}", ha="center", fontsize=7, color="#333333" if i <= t else "#9A9A9A")
    ax_c.set_xlim(0, 16); ax_c.set_ylim(-0.6, 1.25); ax_c.axis("off")
    ax_c.set_title(f"The cache, stored per video: {cache.cache_bytes(t + 1, M, 384) / 1024:g} KiB", fontsize=10)
    buf = io.BytesIO(); fig.savefig(buf, format="png"); plt.close(fig)
    steps.append(Image.open(buf).convert("P", palette=Image.ADAPTIVE))
gif = os.path.join(tempfile.gettempdir(), "vedje_cache.gif")
steps[0].save(gif, save_all=True, append_images=steps[1:] + [steps[-1]] * 3, duration=450, loop=0)
display(ShowImage(filename=gif))

## 3. Follow a point across the frames

Pick a point in the first frame. The frames light up wherever the frozen encoder's features resemble the features around that point, in every sampled frame. VEDJE compresses each frame on its own and keeps the frames in order. Its cache therefore keeps a separate group of tokens for each moment of the video.

In [ ]:
#@title 3. Pick a point in the first frame
x = 50  #@param {type:"slider", min:0, max:100, step:1}
y = 55  #@param {type:"slider", min:0, max:100, step:1}
#@markdown `x` and `y` place the point, in percent of the frame's width and height.
import torch.nn.functional as F
unit = F.normalize(patches[0].float().view(16, 256, -1), dim=-1)
centred = F.normalize(unit - unit.reshape(-1, unit.shape[-1]).mean(0), dim=-1)  # without what all patches share
r, c = min(15, int(y / 100 * 16)), min(15, int(x / 100 * 16))  # the patch under the point, in a 16 x 16 grid
around = [rr * 16 + cc for rr in range(max(0, r - 1), min(16, r + 2)) for cc in range(max(0, c - 1), min(16, c + 2))]
point = F.normalize(centred[0, around].mean(0), dim=0)
similarity = (centred @ point).view(16, 1, 16, 16).cpu()
similarity = F.avg_pool2d(F.pad(similarity, (1, 1, 1, 1), mode="replicate"), 3, stride=1)  # light 3 x 3 smoothing
lo, hi = torch.quantile(similarity, 0.72), torch.quantile(similarity, 0.995)
light = F.interpolate(((similarity - lo) / (hi - lo)).clamp(0, 1) ** 1.3, size=(288, 288), mode="bicubic",
                      align_corners=False).clamp(0, 1)[:, 0].numpy()

fig, axes = plt.subplots(2, 4, figsize=(12, 6.3))
for ax, t in zip(axes.flat, range(0, 16, 2)):
    ax.imshow(frames[t].clamp(0, 1).permute(1, 2, 0).numpy() * (0.18 + 0.82 * light[t])[..., None])
    if t == 0:
        ax.add_patch(plt.Circle(((c + 0.5) * 18, (r + 0.5) * 18), 13, fill=False, edgecolor=BLUE, linewidth=2.5))
        ax.add_patch(plt.Circle(((c + 0.5) * 18, (r + 0.5) * 18), 3, color=BLUE))
    ax.set_title(f"Frame {t + 1}" + (", your point" if t == 0 else ""), fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Time a query on this runtime

At query time, VEDJE reads the query next to each candidate's cache and scores all the candidates in one batch. A typical joint reranker would first encode every candidate's frames again, for every query. Both are measured below on this runtime, with the paper's reranker architecture; the weights here are untrained, which does not change the time. The parameter counts come from Appendix B of the paper, where VEDJE matches the MSR-VTT text-to-video R@1 of the [LamRA](https://arxiv.org/abs/2412.01720) reproduction.

In [ ]:
#@title 4. Time a query: VEDJE against a typical reranker
query = "a peacock walks along a wooden ledge"  #@param {type:"string"}
candidates = 20  #@param {type:"slider", min:5, max:200, step:5}
import contextlib
cfg = config.load_config(os.path.join(REPO, "configs", "vedje_vp_msrvtt.yaml"))
cfg["num_queries_per_frame"] = M
with contextlib.redirect_stdout(io.StringIO()):  # build_model prints its configuration
    net = model.build_model(cfg, training=False).eval().to(device)

# what VEDJE reads for each candidate: the query's tokens and the candidate's cache, in one sequence
ids = net._tokenize_text_vision_pair([query], Z)["input_ids"][0].tolist()
pill = '<span style="display:inline-block;margin:2px;padding:2px 7px;border-radius:9px;background:#E8EAF0;color:#1F2937;font:12px ui-monospace,monospace">{}</span>'
dot = '<span style="display:inline-block;width:7px;height:12px;margin:2px 1px;border-radius:2px;background:#2548F6"></span>'
html, seen = "", 0
for i in ids:
    if i == net.tokenizer.image_token_id:
        html += dot + ('<span style="display:inline-block;width:4px"></span>' if (seen := seen + 1) % M == 0 else "")
    else:
        html += pill.format(net.tokenizer.convert_ids_to_tokens(i))
display(HTML('<div style="font-family:system-ui,sans-serif;color:#4B5563;font-size:13px;margin:4px 0">For each candidate, '
             f'the joint encoder reads the query and the {16 * M} tokens of the cache as one sequence:</div>'
             f'<div style="line-height:22px">{html}</div>'))

def timed(run, repeats=3):
    run()  # warm up
    times = []
    for _ in range(repeats):
        if device == "cuda":
            torch.cuda.synchronize()
        start = time.perf_counter(); run()
        if device == "cuda":
            torch.cuda.synchronize()
        times.append(time.perf_counter() - start)
    return sorted(times)[len(times) // 2]

with torch.inference_mode():
    encode_one = timed(lambda: features.extract_patch_features(backbone, frames), repeats=1)  # one candidate's frames
    caches, prior = Z.expand(candidates, -1, -1).contiguous(), torch.zeros(candidates, device=device)
    vedje_s = timed(lambda: net._chunked_logits([query] * candidates, caches, clip_scores=prior))
typical_s = candidates * encode_one

fig, axes = plt.subplots(1, 3, figsize=(13, 2.7), gridspec_kw={"wspace": 0.12})
panels = [(f"Per query, {candidates} candidates (measured here)", ["Typical reranker: encode the candidates again", "VEDJE: read their caches"],
           [typical_s, vedje_s], [f"{typical_s:.1f} s", f"{vedje_s * 1000:.0f} ms"]),
          ("Stored per video", ["Frame-and-patch features", "VEDJE cache"], [full, stored], [f"{full / 2**20:.2f} MiB", f"{stored / 1024:g} KiB"]),
          ("Online parameters (Appendix B)", ["LamRA reproduction, 7.6B decoder", "VEDJE"], [7.6e9, 157e6], ["7.6B", "157M"])]
for ax, (title, labels, values, texts) in zip(axes, panels):
    top = max(values)
    for yy, label, value, text, colour in zip([1, 0], labels, values, texts, [GRAY, BLUE]):
        ax.barh(yy, max(value, top * 0.006), color=colour, height=0.32)  # the shortest bar keeps a visible sliver
        ax.text(0, yy + 0.27, label, fontsize=9, color="#374151")
        ax.text(max(value, top * 0.006) + top * 0.02, yy, text, va="center", fontsize=10, color=colour, fontweight="bold")
    ax.set_xlim(0, top * 1.3); ax.set_ylim(-0.4, 1.65)
    ax.set_title(title, fontsize=10, loc="left")
    ax.axis("off")
plt.show()
cards(("Typical reranker", f"{typical_s:.1f} s", f"to encode the {candidates} candidates again, before its own joint model runs", GRAY),
      ("VEDJE", f"{vedje_s * 1000:.0f} ms", f"to read their {candidates} caches and score them, {typical_s / vedje_s:,.0f} times less time", BLUE))
display(HTML('<div style="font-family:system-ui,sans-serif;font-size:12px;color:#6B7280;margin:2px 4px">The typical bar counts only '
             'its visual encoder, measured once and multiplied by the number of candidates; a typical reranker also runs its own '
             'joint model. Storage follows Table 9 and the parameter counts Appendix B of the paper.</div>'))

## Use VEDJE on your own videos

Train VEDJE on a GPU, for example on [MSR-VTT](https://openaccess.thecvf.com/content_cvpr_2016/html/Xu_MSR-VTT_A_Large_CVPR_2016_paper.html), then index your videos once and search them with text:

```bash
git clone https://github.com/GabrieleSerussi/vedje.git && cd vedje && pip install -e .
vedje train configs/vedje_vp_msrvtt.yaml
vedje index output/vedje_vp_msrvtt/checkpoint_03.pth my_videos/ --out my_index.pt
vedje search my_index.pt "a person walks across a field"
```

The [README](https://github.com/GabrieleSerussi/vedje#your-data) shows the annotation format and how to use another backbone, and the [project page](https://gabrieleserussi.github.io/vedje/) has the paper's results.

## Cite

```bibtex
@misc{wagner2026vedje,
  title={VEDJE: Video-Efficient Discriminative Joint Encoder for Scalable Video-Text Retrieval},
  author={Shahaf Wagner and Gabriele Serussi and Dan Ben Ami and Tomer Galanti and Chaim Baskin},
  year={2026},
  note={Preprint},
}
```

The sample clip is "Peacock walking and eating" by Mx. Granger on Wikimedia Commons, released under CC0 1.0.